# Seguridad del pipeline de streaming

Endurecemos la ingesta del consumidor de streaming (`tp_mlops2.streaming.consumer`), que
lee eventos del topic `demanda-eventos` y predice online.

**Riesgo:** el consumidor real confÃ­a en todo lo que llega al topic. Un evento con valores
fuera de rango (una temperatura de 900Â°C) o con un productor no autorizado inyectando datos
puede **envenenar** las predicciones en vivo, y si ese stream algÃºn dÃ­a retroalimenta un
reentrenamiento (como el DAG de Airflow de este proyecto, que reentrena periÃ³dicamente),
contaminarÃ­a tambiÃ©n el prÃ³ximo modelo.

**Controles que agregamos:** validaciÃ³n de esquema con rangos fÃ­sicamente plausibles,
cuarentena (DLQ) para eventos que no pasan la validaciÃ³n, y firma HMAC del productor para
que el consumidor pueda verificar que el evento viene de una fuente autorizada.

In [1]:
import hashlib
import hmac
import json
import random

from pydantic import BaseModel, Field, ValidationError

random.seed(0)

CIUDADES = ["Madrid", "Barcelona", "Valencia", "Seville", "Bilbao"]


def evento_real(i: int, envenenado: bool = False) -> dict:
    """Genera un evento con la misma forma real que arma scripts/stream_producer.py."""
    ev = {
        "timestamp": f"2018-06-{(i % 28) + 1:02d}T{(i % 24):02d}:00:00",
        "hour": i % 24,
        "dow": i % 7,
        "month": 6,
        "is_weekend": int((i % 7) >= 5),
        "temp_Madrid": round(random.gauss(25, 5), 1),
        "temp_Barcelona": round(random.gauss(23, 4), 1),
        "temp_Valencia": round(random.gauss(24, 4), 1),
        "temp_Seville": round(random.gauss(30, 5), 1),
        "temp_Bilbao": round(random.gauss(18, 4), 1),
        "load_lag_24h": round(random.gauss(28000, 3000), 1),
        "load_lag_168h": round(random.gauss(28000, 3000), 1),
    }
    if envenenado:
        ev["temp_Madrid"] = 900.0           # temperatura fÃ­sicamente imposible
        ev["load_lag_24h"] = -50000.0       # demanda negativa, imposible
    return ev


stream = [evento_real(i) for i in range(8)] + [evento_real(99, envenenado=True)]
print(f"Stream de prueba: {len(stream)} eventos, el Ãºltimo envenenado")

Stream de prueba: 9 eventos, el Ãºltimo envenenado


## 1. Estado actual: el consumidor acepta todo

`consumer.py` real llama `build_row(message.value)` directo sobre cualquier evento que
llegue al topic, sin validar nada antes de predecir con Ã©l.

In [2]:
# versiÃ³n vulnerable: todo pasa, incluido el evento envenenado
aceptados_v = [e for e in stream]  # build_row() de consumer.py no filtra nada
print(f"Eventos aceptados sin validar: {len(aceptados_v)} de {len(stream)}")
print("el evento envenenado entrÃ³ igual:", any(e.get('temp_Madrid') == 900.0 for e in aceptados_v))

Eventos aceptados sin validar: 9 de 9
el evento envenenado entrÃ³ igual: True


## 2. ValidaciÃ³n de esquema + cuarentena (DLQ) + firma HMAC

In [3]:
SECRETO_PRODUCTOR = b"clave-compartida-del-productor-de-demanda"  # en prod: desde el entorno


def firmar(ev: dict) -> str:
    cuerpo = json.dumps(ev, sort_keys=True).encode()
    return hmac.new(SECRETO_PRODUCTOR, cuerpo, hashlib.sha256).hexdigest()


class EventoDemandaValido(BaseModel):
    """Esquema real de un evento de demanda, con los rangos fÃ­sicamente plausibles
    del dataset (ver notebooks/01_dataset_EDA.ipynb para los rangos observados)."""

    timestamp: str
    hour: int = Field(ge=0, le=23)
    dow: int = Field(ge=0, le=6)
    month: int = Field(ge=1, le=12)
    is_weekend: int = Field(ge=0, le=1)
    temp_Madrid: float = Field(ge=-15, le=50)
    temp_Barcelona: float = Field(ge=-15, le=50)
    temp_Valencia: float = Field(ge=-15, le=50)
    temp_Seville: float = Field(ge=-15, le=50)
    temp_Bilbao: float = Field(ge=-15, le=50)
    load_lag_24h: float = Field(ge=0, le=60000)
    load_lag_168h: float = Field(ge=0, le=60000)


AUDITORIA_PROCEDENCIA: list[dict] = []
DLQ: list[dict] = []


def consumir(ev: dict, firma: str) -> dict | None:
    # 1) autenticaciÃ³n del productor: la firma debe coincidir con el secreto compartido
    if not hmac.compare_digest(firma, firmar(ev)):
        DLQ.append({"evento": ev, "motivo": "firma HMAC invÃ¡lida"})
        return None

    # 2) validaciÃ³n de esquema y rangos
    try:
        validado = EventoDemandaValido(**ev)
    except ValidationError as e:
        DLQ.append({"evento": ev, "motivo": f"esquema invÃ¡lido: {e.errors()[0]['msg']}"})
        return None

    # 3) procedencia: queda registrado quÃ© evento se aceptÃ³ y con quÃ© firma
    AUDITORIA_PROCEDENCIA.append(
        {"timestamp": validado.timestamp, "firma": firma[:12] + "...", "hash_cuerpo": firma}
    )
    return validado.model_dump()


aceptados = []
for i, ev in enumerate(stream):
    # el evento envenenado (Ã­ndice 8) lo mandamos sin firmar correctamente,
    # simulando que no viene de un productor autorizado
    firma_valida = firmar(ev) if i != 8 else "firma-falsa-de-un-productor-no-autorizado"
    resultado = consumir(ev, firma_valida)
    if resultado is not None:
        aceptados.append(resultado)

print(f"Aceptados: {len(aceptados)} de {len(stream)}")
print(f"En cuarentena (DLQ): {len(DLQ)}")
for d in DLQ:
    print(" -", d["motivo"])

Aceptados: 8 de 9
En cuarentena (DLQ): 1
 - firma HMAC invÃ¡lida


## 3. El mismo evento envenenado, pero firmado correctamente

La firma HMAC solo prueba que el evento vino del productor que conoce el secreto â€” no
prueba que los *valores* dentro del evento sean razonables. Un productor autorizado
pero con un sensor roto (o comprometido) puede firmar igual un evento con una temperatura
imposible. Por eso la validaciÃ³n de esquema es un control independiente, no redundante
con la firma.

In [4]:
evento_envenenado = evento_real(99, envenenado=True)
firma_correcta = firmar(evento_envenenado)  # el productor "autorizado" firma algo invÃ¡lido

resultado = consumir(evento_envenenado, firma_correcta)
print("resultado:", resultado, "(None: la validaciÃ³n de esquema lo frenÃ³ igual)")
print("\nDLQ actualizada:")
for d in DLQ:
    print(" -", d["motivo"])

resultado: None (None: la validaciÃ³n de esquema lo frenÃ³ igual)

DLQ actualizada:
 - firma HMAC invÃ¡lida
 - esquema invÃ¡lido: Input should be less than or equal to 50


## 4. PolÃ­tica de alerta sobre la DLQ

Un evento aislado en cuarentena puede ser ruido de sensor. Varios eventos en cuarentena
en poco tiempo son una seÃ±al de que algo sistemÃ¡tico estÃ¡ mal â€” un productor roto, o un
ataque de envenenamiento en curso.

In [5]:
UMBRAL_ALERTA_DLQ = 2  # eventos en cuarentena por ventana para disparar alerta


def chequear_alerta_dlq(dlq_ventana: list[dict]) -> None:
    if len(dlq_ventana) >= UMBRAL_ALERTA_DLQ:
        print(
            f"ALERTA: {len(dlq_ventana)} eventos en cuarentena en esta ventana "
            "- posible ataque de envenenamiento o productor comprometido en curso"
        )
    else:
        print(f"{len(dlq_ventana)} eventos en cuarentena en esta ventana - dentro de lo esperado")


chequear_alerta_dlq(DLQ)

ALERTA: 2 eventos en cuarentena en esta ventana - posible ataque de envenenamiento o productor comprometido en curso


## 5. ReflexiÃ³n

**Â¿Por quÃ© la validaciÃ³n de esquema frena un ataque de envenenamiento?**
Un ataque de envenenamiento no necesita comprometer el canal de comunicaciÃ³n: alcanza con
poder inyectar un evento con valores que empujen las predicciones (o un futuro
reentrenamiento) en una direcciÃ³n deseada. Validar que cada campo estÃ© dentro de un rango
fÃ­sicamente plausible â€” una temperatura entre -15Â°C y 50Â°C, una demanda no negativa â€” corta
ese ataque en el punto de entrada, antes de que el valor llegue al modelo o se acumule en
un dataset de reentrenamiento.

**Â¿QuÃ© agrega la firma HMAC que la validaciÃ³n por sÃ­ sola no da?**
La validaciÃ³n solo mira la *forma* del dato. La firma HMAC prueba la *procedencia*: que el
evento saliÃ³ de un productor que conoce el secreto compartido, no de cualquiera que pueda
escribir al topic. Son complementarias â€” en la celda 3 se ve el caso exacto que separa a
una de la otra: un productor autorizado (firma vÃ¡lida) puede igual mandar datos invÃ¡lidos,
y un atacante sin el secreto puede igual mandar datos con forma correcta.

**Â¿QuÃ© elemento de SAIF cubre este control?**
ValidaciÃ³n de esquema + firma del productor cubre el riesgo especÃ­fico de **envenenamiento
de datos** en un pipeline donde el dato en movimiento eventualmente puede convertirse en
dato de entrenamiento (coherente con el DAG de reentrenamiento de Airflow de este proyecto)
â€” es *adaptar los controles para mitigar amenazas propias de IA*, no un filtro genÃ©rico de
entrada.